In [1]:
from datetime import datetime
from functools import partial
import os
import time

from absl import app, flags, logging
import click
import cv2
import imageio
import jax
import jax.numpy as jnp
import numpy as np

from octo.model.octo_model import OctoModel
from octo.utils.gym_wrappers import HistoryWrapper, TemporalEnsembleWrapper
from octo.utils.train_callbacks import supply_rng

import sys
sys.path.append("../")
from magpie.gripper import Gripper
from magpie import ur5 as ur5
import magpie.realsense_wrapper as real
from magpie.perception import pcd
from magpie.prompt_planner.prompts import mp_prompt_tc_vision as mptc
from PIL import Image

hmpth = "/home/will/workspace/models"
MODEL_CKPT_DICT = {
    "dp": f"{hmpth}/DG.PTH",
    "dp_nf": f"{hmpth}/DGNF.PTH",
    "dp_go": f"{hmpth}/DGGO.PTH",
    "dp_go_nf": f"{hmpth}/DGGONF.PTH",
    "octo_sm_ft": f"{hmpth}/octo_sm_dg",
    "octo_ba_ft": f"{hmpth}/octo_ba_dg",
    "octo_sm_ft_go": f"{hmpth}/octo_sm_dggo",
    "octo_sm": f"hf://rail-berkeley/octo-small-1.5",
    "octo_ba": f"hf://rail-berkeley/octo-base-1.5",
}

OBJECT_NAME = "screwdriver"
CONFIG = {}
CONFIG['vla'] = "octo_sm_ft"
SERVO_PORT = "/dev/ttyACM0"
GRIPPER = None
ROBOT_IP = "192.168.0.4"
VLA_ROBOT = ur5.UR5_Interface(ROBOT_IP)
GRIPPER = Gripper(SERVO_PORT)
CAMERA_SERIAL_INFO = real.poll_devices()
WRIST_CAMERA = real.RealSense(fps=5, w=640, h=480, device_name="D405")
WRIST_CAMERA.initConnection(device_serial=CAMERA_SERIAL_INFO['D405'])
WORKSPACE_CAMERA = real.RealSense(zMax=5, fps=6, w=640, h=480, device_name="D435")
WORKSPACE_CAMERA.initConnection(device_serial=CAMERA_SERIAL_INFO['D435'])

lang_task = f"grasp {OBJECT_NAME} and return home"
sensors = {
    "robot": VLA_ROBOT,
    "gripper": GRIPPER,
    "wrist_camera": WRIST_CAMERA,
    "workspace_camera": WORKSPACE_CAMERA,
    "language_instruction": lang_task,
}

2024-11-14 12:45:57.590140: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-11-14 12:45:57.590167: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-11-14 12:45:57.591260: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-11-14 12:45:58.217117: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
[Open3D INFO] Resetting default logger to print to terminal.
Succeeded to open the port
Succeeded to change the baudrate


In [2]:
# model = OctoModel.load_pretrained(MODEL_CKPT_DICT["octo_sm_ft"], 9999)
model = OctoModel.load_pretrained(MODEL_CKPT_DICT["octo_sm"])

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

tokens.shape=(1, 16, 768)
pad_mask_dict.keys()=dict_keys(['image_primary', 'image_wrist', 'language_instruction', 'timestep'])
keys=('language_instruction',)
key='image_primary'
pad_mask_dict[key].shape=(1,)
key='image_wrist'
pad_mask_dict[key].shape=(1,)
key='language_instruction'
pad_mask_dict[key].shape=(1,)
key='timestep'
pad_mask_dict[key].shape=(1,)
tokens.shape=(1, 2, 256, 512)
pad_mask_dict.keys()=dict_keys(['image_primary', 'image_wrist', 'timestep'])
keys=['image_primary']
key='image_primary'
pad_mask_dict[key].shape=(1, 2)
key='image_wrist'
pad_mask_dict[key].shape=(1, 2)
key='timestep'
pad_mask_dict[key].shape=(1, 2)
tokens.shape=(1, 2, 64, 512)
pad_mask_dict.keys()=dict_keys(['image_primary', 'image_wrist', 'timestep'])
keys=['image_wrist']
key='image_primary'
pad_mask_dict[key].shape=(1, 2)
key='image_wrist'
pad_mask_dict[key].shape=(1, 2)
key='timestep'
pad_mask_dict[key].shape=(1, 2)


In [5]:
instruction = ["pick up the screwdriver"]
task = model.create_tasks(texts=instruction)

In [6]:
def get_observation(sensors, obs_queue, last_obs, first_obs=True, cfg="octo_sm_dg"):
    obs = {}
    
    def process_image(image, size=(128, 128), order=(2, 0, 1)):
        # reshape image from 640x480x3 to 3x480x640 (H, W, C) --> (C, H, W)
        image = np.array(Image.fromarray(image).resize(size))
        image = np.transpose(image, order)
        return image

    wksp_size = (256, 256) if "octo" in cfg else (128, 128)

    joints = sensors["robot"].get_joint_angles()
    tcp = np.array(sensors["robot"].recv.getActualTCPPose())
    gripper_pos = np.array([sensors["gripper"].get_aperture()])/100.0
    applied_force = np.array([sensors["gripper"].applied_force])/100.0
    contact_force = np.array([sensors["gripper"].recorded_contact_force])
    action_blocked = np.array([False])
    obs["image_primary"] = process_image(sensors["workspace_camera"].take_image_blocking(), size=wksp_size, order=(0, 1, 2))
    obs["image_wrist"] = process_image(sensors["wrist_camera"].take_image_blocking(), order=(0, 1, 2))
    obs["timestep_pad_mask"] = False if first_obs else True
    if "ft" in cfg:
        obs["proprio"] = np.concatenate((joints, tcp, gripper_pos, applied_force, contact_force, action_blocked))

    # window=2 so observations with shape (N, ...) become (2, N)
    if first_obs:
        # double the observation
        obs_queue.append({k: np.array([v, v]) for k, v in obs.items()})
    else:
        # take the last_obs and append the new observation to it
        obs_queue.append({k: np.array([last_obs[k], v]) for k, v in obs.items()})

    return obs, obs_queue

In [7]:
OBS_QUEUE = []
LAST_OBS = None

In [8]:
sensors["robot"].start()

Succeeded to open the port
Succeeded to change the baudrate


In [9]:
obs, OBS_QUEUE = get_observation(sensors, OBS_QUEUE, LAST_OBS, first_obs=True, cfg=CONFIG['vla'])
LAST_OBS = obs
batch_window_obs = jax.tree_map(lambda x: x[None], OBS_QUEUE[-1])

In [10]:
for k in OBS_QUEUE[-1]:
    print(f"{k}: {OBS_QUEUE[-1][k].shape}")

image_primary: (2, 256, 256, 3)
image_wrist: (2, 128, 128, 3)
timestep_pad_mask: (2,)


In [11]:
for k in batch_window_obs:
    print(f"{k}: {batch_window_obs[k].shape}")

image_primary: (1, 2, 256, 256, 3)
image_wrist: (1, 2, 128, 128, 3)
timestep_pad_mask: (1, 2)


In [33]:
cfg = CONFIG['vla']
if "ft" in cfg:
    unnorm_stats = model.dataset_statistics["action"]
else:
    unnorm_stats = model.dataset_statistics["berkeley_autolab_ur5"]["action"]
policy = supply_rng(
    partial(
        model.sample_actions,
        unnormalization_statistics=unnorm_stats,
    ),
)


In [34]:
action = policy(
    batch_window_obs, task
)

if "ft" in cfg:
    action = action[:-1]


In [ ]:
def apply_action(actions=[], actuators={}, action_flag="dp", nograsp=False, record_load=False):
    # apply action to actuators
    # actions is a dictionary of action objects
    actions = np.array(actions)
    scale = 1000 # hack for grasp only
    if "go" not in action_flag:
        scale = 100 # need to re-scale the actions
        delta_pos = actions[:3]
        delta_rot = actions[3:6] # not gonna use rotation for now
        actuators["robot"].move_tcp_cartesian_delta(delta_pos, z_offset=0.0)
    
    if nograsp: return

    if "octo" in action_flag and "ft" not in action_flag:
        close_gripper = actions[-1]
        if close_gripper:
            actuators["gripper"].close_gripper()
        else:
            actuators["gripper"].open_gripper()
        return

    curr_aperture = actuators["gripper"].get_aperture()
    if "nf" not in action_flag:
        curr_force = actuators["gripper"].applied_force
        print(f"curr_force: {curr_force}")
        actuators["gripper"].set_force(curr_force + max(actions[-1]*100.0, 0))
        print(f"action: {max(actions[-1]*100.0, 0)}")
        print(f"curr_force after set: {actuators['gripper'].applied_force}")
        actuators["gripper"].set_goal_aperture(curr_aperture + min(actions[-2]*scale, 0), record_load=record_load)
    else:
        actuators["gripper"].set_goal_aperture(curr_aperture + min(actions[-1]*scale, 0), record_load=record_load)


In [ ]:
apply_action(action, actuators=sensors, action_flag=CONFIG['vla'], nograsp=False, record_load=False)

In [36]:
action

Array([[[-6.184e-04, -5.696e-04, -1.626e-03, -6.056e-03,  1.029e-02,
          2.852e-03,  9.893e-01],
        [ 1.180e-03, -6.944e-04, -2.441e-03,  3.784e-03,  1.691e-02,
          1.708e-03,  9.859e-01],
        [ 5.045e-04, -2.313e-04,  1.177e-03, -1.295e-03,  8.253e-03,
          1.443e-03,  9.719e-01],
        [ 3.344e-03, -2.116e-03,  4.606e-03, -6.589e-03,  9.102e-03,
         -2.760e-04,  9.645e-01]]], dtype=float32)

In [37]:
# np.array(action, dtype=np.float64)[0][:6, :-1]
# remove last element of each array in the batch

array([[-2.495e-04, -2.536e-04, -6.960e-04,  6.691e-01,  1.009e-03,
         4.127e-01, -5.122e-01, -6.224e-02],
       [-2.495e-04, -2.759e-04, -6.960e-04, -5.044e-01, -9.856e-04,
         6.359e-01, -5.122e-01, -7.193e-02],
       [-2.495e-04,  2.456e-04, -6.448e-04, -6.280e-01,  1.009e-03,
         4.420e-01,  4.741e-01,  7.484e-02],
       [-2.391e-04, -2.715e-04, -6.960e-04, -5.808e-01,  1.006e-03,
        -1.327e-01,  4.647e-01,  6.940e-02],
       [ 2.618e-05, -2.670e-04, -6.960e-04, -6.635e-01,  7.557e-04,
        -5.760e-01, -5.122e-01,  1.667e-02],
       [-1.921e-04, -2.759e-04, -6.400e-04, -6.635e-01, -9.856e-04,
         4.474e-01,  4.741e-01,  5.117e-02]])

In [17]:
# write batch_window_obs to file
# np.save("batch_window_obs.npy", batch_window_obs)